In [1]:
import os, sys
import pathlib
import json

import torch
import pandas as pd
from transformers import AutoTokenizer
import torch.nn.functional as F

from data.dataset import build_id_to_text
from data.preprocessing import run_preprocessing
from model.train import (
    train_stage1, train_stage2,
    evaluate_epoch, DEFAULT_CFG, format_metrics,
    run_hpo, run_hpo_reranker, build_item_embedding_cache, tokenise_batch
)

from model.architecture import ContrastiveEncoder, TwoTowerModel
from model.reranker import CrossEncoderReranker, TwoTowerWithReranker, train_stage3
from model.metrics import evaluate_reranker, format_metrics

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

In [3]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
output_dir = pathlib.Path('checkpoints/')
output_dir.mkdir(exist_ok=True)

ENCODER = 'intfloat/multilingual-e5-base'
use_tower_hpo = True

# Data

In [4]:
os.makedirs('raw_data', exist_ok=True)

BASE_URL = 'https://huggingface.co/datasets/kdduha/shikimori-recsys/resolve/main'

for fname in ['anime.csv', 'genres.csv', 'users_rates.csv']:
    dest = f'raw_data/{fname}'
    if not os.path.exists(dest):
        !wget -q "{BASE_URL}/{fname}" -O "{dest}"
        print(f'Downloaded {fname}')
    else:
        print(f'Already have {fname}')

!ls -lh raw_data/

Already have anime.csv
Already have genres.csv
Already have users_rates.csv
total 15M
-rw-rw-r-- 1 s.senichev s.senichev 8.7M Mar  2 23:30 anime.csv
-rw-rw-r-- 1 s.senichev s.senichev 2.9K Mar  2 23:30 genres.csv
-rw-rw-r-- 1 s.senichev s.senichev 6.0M Mar  2 23:30 users_rates.csv


In [5]:
stats = run_preprocessing(
    data_dir='raw_data',
    output_dir='processed_data',
)
print(json.dumps(stats, indent=2))

15:53:19  INFO      Loading raw CSVs from raw_data
15:53:19  INFO      Raw sizes — anime: 9950  genres: 80  interactions: 67071
15:53:21  INFO      Anime processing done. Non-null text_input: 9950 / 9950
15:53:21  INFO      Saved anime_processed.parquet
15:53:21  INFO      Saved genre_vocab.json  (80 genres)
15:53:22  INFO      Dropped 0 rows with unparseable anime_id
15:53:22  INFO      Interactions — explicit (scored): 28129  implicit (watched, unscored): 38677  dropped (score=0, episodes=0): 265
15:53:23  INFO      Dropped 1515 interactions for unknown anime_id
15:53:23  INFO      Removed users with < 3 explicit ratings. Kept: 531 users  39345 rows
15:53:23  INFO      Final — 39345 rows (27214 explicit, 12131 implicit)  531 users  3695 items
15:53:23  INFO      Temporal split — train: 38393  val: 476  test: 476  (eligible users: 476 / 531)
15:53:23  WARNING   Temporal leakage: 130 val rows have created_at BEFORE user's latest train item (likely duplicate timestamps)
15:53:23  WARNIN

{
  "n_anime_raw": 9950,
  "n_anime_processed": 9950,
  "n_genres": 80,
  "n_interactions_raw": 67071,
  "n_interactions_clean": 39345,
  "n_users": 531,
  "n_items": 3695,
  "train_size": 38393,
  "val_size": 476,
  "test_size": 476,
  "score_mean": 5.498,
  "score_std": 3.939,
  "density_pct": 2.0053
}


In [7]:
anime_df = pd.read_parquet('processed_data/anime_processed.parquet')
train_df = pd.read_parquet('processed_data/train_interactions.parquet')
val_df = pd.read_parquet('processed_data/val_interactions.parquet')
test_df = pd.read_parquet('processed_data/test_interactions.parquet')

print('Anime sample')
display(anime_df[['id','name','genre_names','rating_ordinal','score_global','text_input']].head(3))

print('\nInteraction sample')
display(train_df.head(3))

print(f'\nTrain: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}')
# print(f'Score distribution:\n{train_df.score_raw.value_counts().sort_index()}')

Anime sample


,id,name,genre_names,rating_ordinal,score_global,text_input
0,52991,Sousou no Frieren,"[Adventure, Drama, Fantasy, Shounen]",2.0,0.921111,Sousou no Frieren (Провожающая в последний пут...
1,59978,Sousou no Frieren 2nd Season,"[Adventure, Drama, Fantasy, Shounen]",2.0,0.910000,Sousou no Frieren 2nd Season (Провожающая в по...
2,5114,Fullmetal Alchemist: Brotherhood,"[Action, Adventure, Drama, Fantasy, Shounen, M...",3.0,0.900000,Fullmetal Alchemist: Brotherhood (Стальной алх...



Interaction sample


,user_id,anime_id,score_raw,score_norm,rewatches,episodes,completion_rate,confidence,is_explicit,created_at
0,1721273,5114,9,0.888889,0,64,1.0,0.888889,True,2025-11-07 19:14:34+00:00
1,1721273,9253,10,1.000000,0,24,1.0,1.000000,True,2025-11-07 19:15:22+00:00
2,1721273,2001,10,1.000000,0,27,1.0,1.000000,True,2025-11-07 19:18:22+00:00



Train: 38,393  Val: 476  Test: 476


# Model

### Load checkpoint if it exists

In [11]:
# output_dir = pathlib.Path('checkpoints/final_model_v4_with_rerank_hpo')

# with open(output_dir / 'config.json') as f:
#     cfg = json.load(f)

# tokenizer = AutoTokenizer.from_pretrained(str(output_dir))

# model = TwoTowerModel(
#     encoder_name=ENCODER,
#     proj_dim=cfg['proj_dim'],
#     nhead=cfg['nhead'],
#     temperature=cfg['temperature'],
#     dropout=cfg['dropout'],
#     freeze_mode=cfg['freeze_mode'],
#     lora_rank=cfg['lora_rank'],
#     lora_alpha=cfg.get('lora_alpha', 16.0),
# )

# state = torch.load(output_dir / 'model.pt', map_location=device)
# model.load_state_dict(state)
# model.to(device)
# model.eval()
# print('Final model loaded')

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


[ItemTower] Applied LoRA to 24 linear layers
[ItemTower] Gradient checkpointing enabled
Final model loaded


## Train main model

### HPO

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(ENCODER)
hpo_path = output_dir / 'hpo_best_params.json'

if use_tower_hpo:
    print('Starting HPO')
    best_params = run_hpo(
        train_df=train_df,
        val_df=val_df,
        anime_df=anime_df,
        base_cfg=DEFAULT_CFG,
        output_dir=pathlib.Path('checkpoints'),
        device=device,
        n_trials=20,
        encoder_name=ENCODER,
    )
    
    with open(hpo_path) as f:
        hpo_data = json.load(f)
    cfg = {**DEFAULT_CFG, **hpo_data['best_params']}
    print(f'Using HPO params (NDCG@10={hpo_data["best_value"]:.4f})')

else:
    cfg = {**DEFAULT_CFG}
    print('Using default hyperparameters')

print(json.dumps(cfg, indent=2))

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Starting HPO


15:54:19  INFO      Starting HPO  (n_trials=20, hpo_epochs_per_trial=6)
15:54:20  INFO      ============================================================
15:54:20  INFO      Stage 2: Two-tower training
15:54:20  INFO      ============================================================


[ItemTower] Applied LoRA to 24 linear layers
[ItemTower] Gradient checkpointing enabled


/home/s.senichev/anirec/model/train.py:344: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device.type == "cuda"))
15:54:34  INFO        Building item embedding cache...
/home/s.senichev/anirec/model/train.py:240: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can

...

/home/s.senichev/anirec/model/train.py:344: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device.type == "cuda"))
16:14:31  INFO        Building item embedding cache...
/home/s.senichev/anirec/model/train.py:240: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can

[ItemTower] Applied LoRA to 24 linear layers
[ItemTower] Gradient checkpointing enabled


/home/s.senichev/anirec/model/train.py:344: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device.type == "cuda"))
16:19:24  INFO        Building item embedding cache...
/home/s.senichev/anirec/model/train.py:240: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can

[ItemTower] Applied LoRA to 24 linear layers
[ItemTower] Gradient checkpointing enabled


/home/s.senichev/anirec/model/train.py:344: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler(enabled=(device.type == "cuda"))
16:24:36  INFO        Building item embedding cache...
/home/s.senichev/anirec/model/train.py:240: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=(device.type == "cuda")):
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can

### Train Stage 1 (item tower)

In [ ]:
s1_model = ContrastiveEncoder(
    encoder_name=ENCODER,
    proj_dim=cfg['proj_dim'],
    dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'],
    lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
)

s1_model = train_stage1(
    s1_model, tokenizer,
    train_df=train_df,
    val_df=val_df,
    anime_df=anime_df,
    cfg=cfg,
    output_dir=output_dir,
    device=device,
)

torch.save(s1_model.tower.state_dict(), output_dir / 'stage1_encoder.pt')
print('Stage 1 complete')

### Train Stage 2 (user tower)

In [ ]:
model = TwoTowerModel(
    encoder_name=ENCODER,
    proj_dim=cfg['proj_dim'],
    nhead=cfg['nhead'],
    temperature=cfg['temperature'],
    dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'],
    lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
)

state = torch.load(output_dir / 'stage1_encoder.pt', map_location='cpu')
model.item_tower.load_state_dict(state, strict=False)
print('Stage 1 weights loaded into item tower')

model, best_val = train_stage2(
    model, tokenizer,
    train_df=train_df,
    val_df=val_df,
    anime_df=anime_df,
    cfg=cfg,
    output_dir=output_dir,
    device=device,
)

print('Best Val Metrics:', format_metrics(best_val))

### Save model

In [ ]:
final_dir = pathlib.Path('checkpoints/final_model_v4_with_rerank_hpo')
final_dir.mkdir(exist_ok=True)

torch.save(model.state_dict(), final_dir / 'model.pt')
tokenizer.save_pretrained(str(final_dir))
with open(final_dir / 'config.json', 'w') as f:
    json.dump(cfg, f, indent=2)

print(f'Model saved to {final_dir}')

## Evaluate model

In [ ]:
train_and_val = pd.concat([train_df, val_df], ignore_index=True)
test_metrics = evaluate_epoch(
    model, tokenizer, test_df, anime_df,
    cfg=cfg, device=device, ks=[10, 20, 50],
    train_df=train_and_val,
)

print('TEST RESULTS')
for k, v in sorted(test_metrics.items()):
    print(f'{k:<12} {v:.4f}')

## Reranker model

### Load checkpoint if it exists

In [36]:
# cfg = {**DEFAULT_CFG}

# s3_tokenizer = AutoTokenizer.from_pretrained(cfg["s3_encoder"])

# reranker_model = CrossEncoderReranker(encoder_name=cfg['s3_encoder'])
# reranker_model.load_state_dict(
#     torch.load('checkpoints/stage3_best.pt', map_location='cpu')
# )
# reranker_model.to(device).eval()
# print("Reranker loaded")

### Train reranker with HPO

In [ ]:
from model.train import run_hpo_reranker, DEFAULT_CFG

best_s3_params = run_hpo_reranker(
    two_tower_model=model,
    train_df=train_df,
    val_df=val_df,
    anime_df=anime_df,
    base_cfg=cfg,
    output_dir=output_dir,
    device=device,
    n_trials=20,
)
print(best_s3_params)

cfg_s3 = {**cfg, **best_s3_params}
reranker_model = train_stage3(
    CrossEncoderReranker(encoder_name=cfg['s3_encoder']),
    s3_tokenizer, train_df, val_df, anime_df,
    cfg=cfg_s3, output_dir=output_dir, device=device,
)

### Save reranker model

In [ ]:
torch.save(reranker_model.state_dict(), final_dir / 'reranker.pt')
s3_tokenizer.save_pretrained(str(final_dir / 'reranker_tokenizer'))

print(f'Reranker saved to {final_dir}')

### Evaludate reranker

In [ ]:
val_metrics = evaluate_reranker(
    recommender=recommender,
    holdout_df=val_df,
    train_df=train_df,
    ks=[10, 20, 50],
    retrieval_k=100,
)
print("RERANKER VAL METRICS")
print(format_metrics(val_metrics))

In [ ]:
# Test set (context = train + val)
import pandas as pd
train_and_val = pd.concat([train_df, val_df], ignore_index=True)
test_metrics = evaluate_reranker(
    recommender=recommender,
    holdout_df=test_df,
    train_df=train_and_val,
    ks=[10, 20, 50],
    retrieval_k=100,
)
print("RERANKER TEST METRICS")
print(format_metrics(test_metrics))

# Load fully trained model

In [ ]:
final_dir = pathlib.Path('checkpoints/final_model_v4_with_rerank_hpo')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

with open(final_dir / 'config.json') as f:
    cfg = json.load(f)

tokenizer = AutoTokenizer.from_pretrained(str(final_dir))
model = TwoTowerModel(
    encoder_name='intfloat/multilingual-e5-base',
    proj_dim=cfg['proj_dim'], nhead=cfg['nhead'],
    temperature=cfg['temperature'], dropout=cfg['dropout'],
    freeze_mode=cfg['freeze_mode'], lora_rank=cfg['lora_rank'],
    lora_alpha=cfg.get('lora_alpha', 16.0),
)
model.load_state_dict(torch.load(final_dir / 'model.pt', map_location='cpu'))
model.to(device).eval()

s3_tokenizer = AutoTokenizer.from_pretrained(str(final_dir / 'reranker_tokenizer'))
reranker_model = CrossEncoderReranker(encoder_name=cfg['s3_encoder'])
reranker_model.load_state_dict(torch.load(final_dir / 'reranker.pt', map_location='cpu'))
reranker_model.to(device).eval()

print('Loaded')

# Inference 

In [ ]:
# sample user

user_history = [
    (5114, 10),   # FMA Brotherhood
    (9253, 9),   # Steins;Gate
    (11061, 9),   # HxH 2011
    (37510, 8),   # Mob Psycho 100 II
    (31240, 3),   # Some disliked item (placeholder)
]
top_k = 10

## Test model without reranker

In [ ]:
model.eval()

item_matrix, id_to_idx = build_item_embedding_cache(model, tokenizer, anime_df, cfg, device)
idx_to_id = {v: k for k, v in id_to_idx.items()}

context_ids = [aid  for aid, _ in user_history if aid in id_to_idx]
context_scores = [s/10 for aid, s  in user_history if aid in id_to_idx]
watched_ids = set(context_ids)

ctx_idxs = torch.tensor([id_to_idx[a] for a in context_ids], dtype=torch.long)
ctx_embs = item_matrix[ctx_idxs].unsqueeze(0).to(device)
ctx_scores= torch.tensor([context_scores], dtype=torch.float).to(device)
ctx_mask  = torch.ones(1, len(context_ids), dtype=torch.bool).to(device)

with torch.no_grad():
    user_vec = model.encode_user(ctx_embs, ctx_scores, ctx_mask)
    scores = (user_vec @ item_matrix.to(device).T).squeeze(0)

# Exclude watched items
watched_idxs = set(id_to_idx[a] for a in watched_ids if a in id_to_idx)
for idx in watched_idxs:
    scores[idx] = -1e9

top_idxs = scores.topk(top_k).indices.cpu().tolist()
top_ids = [idx_to_id[i] for i in top_idxs]

print(f'Top-{top_k} recommendations for this user:\n')
id_to_row = anime_df.set_index('id')
for rank, aid in enumerate(top_ids, 1):
    row = id_to_row.loc[aid]
    genres = ', '.join(row['genre_names'][:3])
    print(f'  {rank:2}. {row["name"]:<45} [{genres}]  score={row["score_global"]:.2f}')

## Test model with reranker

In [ ]:
recommender = TwoTowerWithReranker(
    two_tower=model,
    reranker=reranker_model,
    tokenizer=tokenizer,
    reranker_tokenizer=s3_tokenizer,
    anime_df=anime_df,
    device=device,
)

recs = recommender.recommend(user_history, top_k=top_k, retrieval_k=100)

print(f'Top- recommendations\n')
print(f'  {"#":<4} {"Title":<45} {"Reranker":>10} {"Retrieved":>10} {"Genres"}')
print('  ' + '-'*85)
for r in recs:
    genres = ', '.join(list(r['genres'])[:3]) if len(r['genres']) > 0 else '—'
    moved = r['two_tower_rank'] - r['rank']
    arrow = f'↑{moved}' if moved > 0 else (f'↓{-moved}' if moved < 0 else '=')
    print(f"  {r['rank']:<4} {r['name']:<45} {r['reranker_score']:>6.3f}  "
          f"#{r['two_tower_rank']:<3} {arrow:<5}  [{genres}]")